# Demand Forecasting
Predict daily, weekly, and monthly sales using traditional, machine-learning, and deep-learning approaches.


## 1. Business Problem
Forecast future sales to support inventory and demand planning.


## 2. Business Objectives
- Predict daily sales
- Summarise weekly and monthly demand
- Compare forecasting models using RMSE, MAE, and MAPE


## 3. ML Workflow
Data Preparation ? Time-Series Features ? Forecasting Models ? Evaluation ? Demand Prediction


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from prophet import Prophet
except ImportError:
    Prophet = None

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from sklearn.preprocessing import MinMaxScaler
except ImportError:
    tf = None


## 4. Load Dataset


In [ ]:
DATA_PATH = Path("../data/processed/master_df.parquet")
master_df = pd.read_parquet(DATA_PATH)
master_df.head()


## 5. Dataset Overview


In [ ]:
print("Dataset shape:", master_df.shape)
master_df[["order_purchase_timestamp", "payment_value"]].info()


## 6. Data Quality Check


In [ ]:
display(master_df[["order_purchase_timestamp", "payment_value"]].isna().sum().to_frame("Missing Values"))


## 7. Missing Values Handling


In [ ]:
demand_df = master_df[["order_purchase_timestamp", "payment_value"]].dropna().copy()
demand_df["order_purchase_timestamp"] = pd.to_datetime(demand_df["order_purchase_timestamp"])
demand_df["payment_value"] = demand_df["payment_value"].fillna(0)


## 8. Duplicate Analysis


In [ ]:
print("Duplicate transaction rows:", demand_df.duplicated().sum())


## 9. Demand Definition


Daily demand is defined as total customer payment value per purchase date. The forecast target is future daily sales revenue.


## 10. Daily Sales Feature Engineering


In [ ]:
daily_sales = (
    demand_df.set_index("order_purchase_timestamp")
    .resample("D")["payment_value"]
    .sum()
    .rename("Sales")
    .asfreq("D", fill_value=0)
    .to_frame()
)

daily_sales["day_of_week"] = daily_sales.index.dayofweek
daily_sales["month"] = daily_sales.index.month
for lag in [1, 7, 14]:
    daily_sales[f"lag_{lag}"] = daily_sales["Sales"].shift(lag)
daily_sales["rolling_mean_7"] = daily_sales["Sales"].shift(1).rolling(7).mean()
model_df = daily_sales.dropna().copy()
display(model_df.head())


## 11. Daily Sales Trend


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily_sales.index, daily_sales["Sales"])
plt.title("Daily Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()


## 12. Weekly Sales


In [ ]:
weekly_sales = daily_sales["Sales"].resample("W").sum()
plt.figure(figsize=(12, 5))
plt.plot(weekly_sales.index, weekly_sales)
plt.title("Weekly Sales")
plt.show()
display(weekly_sales.tail().to_frame("Weekly Sales"))


## 13. Monthly Sales


In [ ]:
monthly_sales = daily_sales["Sales"].resample("ME").sum()
plt.figure(figsize=(12, 5))
plt.plot(monthly_sales.index, monthly_sales)
plt.title("Monthly Sales")
plt.show()
display(monthly_sales.tail().to_frame("Monthly Sales"))


## 14. Train/Test Split


In [ ]:
test_days = min(30, max(7, len(model_df) // 5))
train = model_df.iloc[:-test_days].copy()
test = model_df.iloc[-test_days:].copy()

X_train = train.drop(columns="Sales")
y_train = train["Sales"]
X_test = test.drop(columns="Sales")
y_test = test["Sales"]
print("Training days:", len(train))
print("Testing days:", len(test))


## 15. Evaluation Function


In [ ]:
def evaluate_forecast(name, actual, prediction):
    prediction = np.maximum(np.asarray(prediction), 0)
    result = {
        "Model": name,
        "RMSE": mean_squared_error(actual, prediction) ** 0.5,
        "MAE": mean_absolute_error(actual, prediction),
        "MAPE": mean_absolute_percentage_error(actual, prediction) * 100
    }
    return result, prediction

forecast_results = []
forecast_predictions = {}


## 16. ARIMA


In [ ]:
arima_model = ARIMA(train["Sales"], order=(1, 1, 1)).fit()
arima_forecast = arima_model.forecast(steps=len(test))
result, arima_forecast = evaluate_forecast("ARIMA", y_test, arima_forecast)
forecast_results.append(result)
forecast_predictions["ARIMA"] = arima_forecast
print(result)


## 17. SARIMA


In [ ]:
sarima_model = SARIMAX(train["Sales"], order=(1, 1, 1), seasonal_order=(1, 0, 1, 7)).fit(disp=False)
sarima_forecast = sarima_model.forecast(steps=len(test))
result, sarima_forecast = evaluate_forecast("SARIMA", y_test, sarima_forecast)
forecast_results.append(result)
forecast_predictions["SARIMA"] = sarima_forecast
print(result)


## 18. Prophet


In [ ]:
if Prophet is not None:
    prophet_train = train.reset_index()[["order_purchase_timestamp", "Sales"]].rename(columns={"order_purchase_timestamp": "ds", "Sales": "y"})
    prophet_model = Prophet(daily_seasonality=True, weekly_seasonality=True)
    prophet_model.fit(prophet_train)
    future = prophet_model.make_future_dataframe(periods=len(test), freq="D")
    prophet_forecast = prophet_model.predict(future).tail(len(test))["yhat"].values
    result, prophet_forecast = evaluate_forecast("Prophet", y_test, prophet_forecast)
    forecast_results.append(result)
    forecast_predictions["Prophet"] = prophet_forecast
    print(result)
else:
    print("Prophet is not installed; skip this cell or install prophet to run it.")


## 19. XGBoost


In [ ]:
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_forecast = xgb_model.predict(X_test)
result, xgb_forecast = evaluate_forecast("XGBoost", y_test, xgb_forecast)
forecast_results.append(result)
forecast_predictions["XGBoost"] = xgb_forecast
print(result)


## 20. LightGBM


In [ ]:
lgbm_model = LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=42, verbosity=-1)
lgbm_model.fit(X_train, y_train)
lgbm_forecast = lgbm_model.predict(X_test)
result, lgbm_forecast = evaluate_forecast("LightGBM", y_test, lgbm_forecast)
forecast_results.append(result)
forecast_predictions["LightGBM"] = lgbm_forecast
print(result)


## 21. LSTM


In [ ]:
if tf is not None:
    scaler = MinMaxScaler()
    scaled_sales = scaler.fit_transform(daily_sales[["Sales"]]).flatten()
    window = 14
    X_seq, y_seq = [], []
    for i in range(window, len(scaled_sales)):
        X_seq.append(scaled_sales[i-window:i])
        y_seq.append(scaled_sales[i])
    X_seq, y_seq = np.array(X_seq), np.array(y_seq)
    split = len(X_seq) - len(test)
    lstm_model = Sequential([LSTM(32, input_shape=(window, 1)), Dense(1)])
    lstm_model.compile(optimizer="adam", loss="mse")
    lstm_model.fit(X_seq[:split].reshape(-1, window, 1), y_seq[:split], epochs=10, batch_size=16, verbose=0)
    lstm_forecast = scaler.inverse_transform(lstm_model.predict(X_seq[split:].reshape(-1, window, 1), verbose=0)).flatten()
    result, lstm_forecast = evaluate_forecast("LSTM", y_test, lstm_forecast)
    forecast_results.append(result)
    forecast_predictions["LSTM"] = lstm_forecast
    print(result)
else:
    print("TensorFlow is not installed; skip this cell or install tensorflow to run it.")


## 22. Model Comparison


In [ ]:
comparison = pd.DataFrame(forecast_results).sort_values("RMSE")
display(comparison)
best_model_name = comparison.iloc[0]["Model"]
print(f"Best model by RMSE: {best_model_name}")


## 23. Forecast Plot


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(test.index, y_test, label="Actual", linewidth=2)
for name, prediction in forecast_predictions.items():
    plt.plot(test.index, prediction, label=name, alpha=0.75)
plt.title("Demand Forecast Comparison")
plt.xlabel("Date")
plt.ylabel("Daily Sales")
plt.legend()
plt.show()


## 24. Error Metrics


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=comparison, x="RMSE", y="Model")
plt.title("RMSE by Forecasting Model")
plt.show()
display(comparison[["Model", "RMSE", "MAE", "MAPE"]])


## 25. Daily Demand Prediction


In [ ]:
demand_prediction = pd.DataFrame({
    "Date": test.index,
    "Actual_Sales": y_test.values,
    "Predicted_Sales": forecast_predictions[best_model_name]
})
display(demand_prediction.tail(10))


## 26. Weekly Demand Prediction


In [ ]:
weekly_prediction = demand_prediction.set_index("Date").resample("W")[["Actual_Sales", "Predicted_Sales"]].sum()
display(weekly_prediction)


## 27. Monthly Demand Prediction


In [ ]:
monthly_prediction = demand_prediction.set_index("Date").resample("ME")[["Actual_Sales", "Predicted_Sales"]].sum()
display(monthly_prediction)


## 28. Demand Insights


In [ ]:
print(f"Selected model: {best_model_name}")
print(f"Average predicted daily demand: {demand_prediction['Predicted_Sales'].mean():.2f}")
print(f"Peak predicted daily demand: {demand_prediction['Predicted_Sales'].max():.2f}")


## 29. Inventory Recommendation


In [ ]:
recommended_daily_stock = demand_prediction["Predicted_Sales"].mean() * 1.10
print(f"Suggested daily inventory target with 10% buffer: {recommended_daily_stock:.2f}")


## 30. Model Export


In [ ]:
model_to_export = {"XGBoost": xgb_model, "LightGBM": lgbm_model}.get(best_model_name)
if model_to_export is not None:
    import joblib
    joblib.dump(model_to_export, "demand_forecasting_model.joblib")
    print(f"{best_model_name} model exported.")
else:
    print("Traditional and deep-learning models are not exported in this simple workflow.")


## 31. Executive Summary


In [ ]:
print("Demand forecasting workflow completed.")
print(f"Best model: {best_model_name}")
print(f"Best RMSE: {comparison.iloc[0]['RMSE']:.2f}")
print("Use daily, weekly, and monthly predictions for inventory planning.")
